In [6]:
import pandas as pd
import numpy as np
import importlib
import statsmodels.formula.api as smf
%load_ext rpy2.ipython

from statsmodels.genmod.families import Gaussian
from statsmodels.genmod.cov_struct import Autoregressive, Unstructured

import analysis
importlib.reload(analysis)

<module 'analysis' from 'c:\\Users\\JoleneWium\\Documents\\Projects\\biostats\\PPH7095F\\20260402\\code\\analysis.py'>

In [7]:
df = analysis.load_data()

In [8]:
dat = df[
    [
        "id",
        "obs_time",
        "satisfaction",
        "sex",
        "age_marriage",
        "cohab",
        "income",
        "hw_all",
    ]
].dropna().copy()

dat = dat.sort_values(["id", "obs_time"]).reset_index(drop=True)

# visit order within person
dat["time_index"] = dat.groupby("id").cumcount().astype(int)

formula = (
    "satisfaction ~ time_index + sex + age_marriage + "
    "cohab + income + hw_all"
)

In [9]:
# LME 1: Random intercept only
lme1 = smf.mixedlm(
    formula=formula,
    data=dat,
    groups=dat["id"],   # random intercept
).fit()
lme1.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
          Mixed Linear Model Regression Results
==========================================================
Model:            MixedLM Dependent Variable: satisfaction
No. Observations: 3813    Method:             REML        
No. Groups:       494     Scale:              0.8455      
Min. group size:  3       Log-Likelihood:     -5786.3216  
Max. group size:  17      Converged:          Yes         
Mean group size:  7.7                                     
----------------------------------------------------------
               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept       6.968    0.425  16.397 0.000  6.135  7.801
sex[T.male]    -0.434    0.153  -2.837 0.005 -0.734 -0.134
time_index     -0.308    0.006 -55.016 0.000 -0.319 -0.297
age_marriage    0.008    0.013   0.648 0.517 -0.017  0.033
cohab           0.262    0.178   1.473 0.141 -0.087  0.612
income         -0.000    0.000  -0.384 0.701 -0.000  0.000
hw_all         -0.579    0.152  -3.815 0.000 -0.876 -0.281
Group Var       1.693    0.136                            
==========================================================

"""

In [10]:
lme1_fnc, lme2_fnc = analysis.fit_lme_models(df)

In [25]:
print(lme1_fnc.summary())

          Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: satisfaction
No. Observations: 3813    Method:             ML          
No. Groups:       494     Scale:              0.8453      
Min. group size:  3       Log-Likelihood:     -5762.0894  
Max. group size:  17      Converged:          Yes         
Mean group size:  7.7                                     
----------------------------------------------------------
               Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------
Intercept       6.968    0.422  16.499 0.000  6.140  7.795
sex[T.male]    -0.434    0.152  -2.855 0.004 -0.732 -0.136
obs_time       -0.308    0.006 -55.016 0.000 -0.319 -0.297
age_marriage    0.008    0.013   0.652 0.514 -0.017  0.033
cohab           0.262    0.177   1.483 0.138 -0.085  0.609
income         -0.000    0.000  -0.388 0.698 -0.000  0.000
hw_all         -0.579    0.151  -3.839 0.000 -0.874 -0.283
Group Va

In [12]:
# LME 2: Random intercept + random slope for time
lme2 = smf.mixedlm(
    formula=formula,
    data=dat,
    groups=dat["id"],
    re_formula="~time_index",  # adds random slope
).fit()
lme2.summary()

<class 'statsmodels.iolib.summary2.Summary'>
"""
              Mixed Linear Model Regression Results
==================================================================
Model:                MixedLM   Dependent Variable:   satisfaction
No. Observations:     3813      Method:               REML        
No. Groups:           494       Scale:                0.7607      
Min. group size:      3         Log-Likelihood:       -5738.6336  
Max. group size:      17        Converged:            Yes         
Mean group size:      7.7                                         
------------------------------------------------------------------
                       Coef.  Std.Err.    z    P>|z| [0.025 0.975]
------------------------------------------------------------------
Intercept               6.920    0.427  16.221 0.000  6.084  7.756
sex[T.male]            -0.455    0.153  -2.970 0.003 -0.755 -0.155
time_index             -0.342    0.009 -39.183 0.000 -0.359 -0.325
age_marriage            0.011    0.013   0.819 0.413 -0.015  0.036
cohab                   0.298    0.179   1.669 0.095 -0.052  0.648
income                 -0.000    0.000  -0.217 0.829 -0.000  0.000
hw_all                 -0.542    0.152  -3.567 0.000 -0.840 -0.244
Group Var               1.659    0.155                            
Group x time_index Cov -0.006    0.014                            
time_index Var          0.012    0.003                            
==================================================================

"""

In [13]:
print(lme2_fnc.summary())

             Mixed Linear Model Regression Results
Model:              MixedLM   Dependent Variable:   satisfaction
No. Observations:   3813      Method:               ML          
No. Groups:         494       Scale:                0.7609      
Min. group size:    3         Log-Likelihood:       -5714.7692  
Max. group size:    17        Converged:            Yes         
Mean group size:    7.7                                         
----------------------------------------------------------------
                     Coef.  Std.Err.    z    P>|z| [0.025 0.975]
----------------------------------------------------------------
Intercept             6.920    0.424  16.322 0.000  6.089  7.751
sex[T.male]          -0.455    0.152  -2.988 0.003 -0.753 -0.157
obs_time             -0.341    0.009 -39.225 0.000 -0.359 -0.324
age_marriage          0.011    0.013   0.823 0.410 -0.015  0.036
cohab                 0.298    0.177   1.679 0.093 -0.050  0.646
income               -0.000    0.000  -

In [14]:
%R -i df

In [15]:
import rpy2
import rpy2.robjects as ro
print(ro.r("R.version.string"))

[1] "R version 4.5.1 (2025-06-13 ucrt)"



In [16]:
%%R
head(df)

   id obs_time satisfaction    sex age_marriage cohab  income hw_all
0   1       14          6.7 female           32     1 39110.4      0
1   1       78          5.2 female           32     1 39110.4      0
2   1      163          5.2 female           32     1 39110.4      0
13 23       63          6.6 female           32     1 92140.8      0
14 23       97          6.3 female           32     1 92140.8      0
15 23      114          7.2 female           32     1 92140.8      0


In [23]:
%%R
install.packages("lme4", repos = "https://cloud.r-project.org")

Warning in install.packages("lme4", repos = "https://cloud.r-project.org") :
  'lib = "C:/Program Files/R/R-4.5.1/library"' is not writable
Error in install.packages("lme4", repos = "https://cloud.r-project.org") : 
  unable to install packages


RInterpreterError: Failed to parse and evaluate line 'install.packages("lme4", repos = "https://cloud.r-project.org")\n'.
R error message: 'Error in install.packages("lme4", repos = "https://cloud.r-project.org") : \n  unable to install packages'
R stdout:
Warning in install.packages("lme4", repos = "https://cloud.r-project.org") :
  'lib = "C:/Program Files/R/R-4.5.1/library"' is not writable
Error in install.packages("lme4", repos = "https://cloud.r-project.org") : 
  unable to install packages

In [21]:
%%R -i df
dat <- df[, c(
  "id",
  "obs_time",
  "satisfaction",
  "sex",
  "age_marriage",
  "cohab",
  "income",
  "hw_all"
)]

dat <- dat[order(dat$id, dat$obs_time), ]

library(lme4)

lme2 <- lmer(
  satisfaction ~ obs_time + sex + age_marriage +
    cohab + income + hw_all +
    (time_index | id),
  data = dat
)

summary(lme2)


Error in library(lme4) : there is no package called 'lme4'


RInterpreterError: Failed to parse and evaluate line 'dat <- df[, c(\n  "id",\n  "obs_time",\n  "satisfaction",\n  "sex",\n  "age_marriage",\n  "cohab",\n  "income",\n  "hw_all"\n)]\n\ndat <- dat[order(dat$id, dat$obs_time), ]\n\nlibrary(lme4)\n\nlme2 <- lmer(\n  satisfaction ~ obs_time + sex + age_marriage +\n    cohab + income + hw_all +\n    (time_index | id),\n  data = dat\n)\n\nsummary(lme2)\n'.
R error message: "Error in library(lme4) : there is no package called 'lme4'"